# Better ensembles by mixing ROMAN scales

A five-model ensemble can gain more from complementary representations than from five random initializations of the same input. On the paper's FaceFour dataset, the mixed-scale MiniRocket ensemble correctly classifies all 88 test series, compared with 83 for the baseline ensemble.

FaceFour is a quick example: only 24 training series, 88 test series and 350 time steps. We reproduce the paper's five-member hard-voting comparison:

- **Baseline:** S=1 with seeds 0, 1, 2, 3 and 4.
- **Mixed scales:** S=1 with seeds 0 and 1, plus S=2, S=3 and S=4 with seed 0.

Both ensembles contain five members with 10,000 MiniRocket kernels each. Shared baseline members are fitted once, so only eight unique fits are needed. The scale and seed choices are fixed before running this notebook.

In [1]:
from pathlib import Path
import sys

# Support running the notebook from the repository root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "roman" / "__init__.py").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "notebooks"))

from IPython.display import display
from example_utils import (load_dataset, make_model, benchmark_single,
                           benchmark_ensembles, display_single, environment)


In [2]:
DATASET = "FaceFour"
DATA_ROOT = None  # Optional local cache; first use may download the dataset.
X_train, y_train, X_test, y_test = load_dataset(DATASET, DATA_ROOT)
print("Training:", X_train.shape, "  Test:", X_test.shape)

Training: (24, 1, 350)   Test: (88, 1, 350)


## Train and compare

Each member predicts directly from the original test inputs. Hard voting uses the training-label class order to resolve ties. Training time is the sum of the five member fit times; inference includes all five complete member predictions plus voting. Both paths are warmed, use one compute thread and are timed on the same test batch.

In [3]:
results, predictions = benchmark_ensembles(
    X_train, y_train, X_test, y_test, repeats=5,
)
view = results.rename(columns={"ensemble": "Ensemble", "members": "Members",
                              "test_accuracy": "Test accuracy", "correct": "Correct",
                              "training_s": "Training time (s)", "inference_s": "Inference time (s)"})
display(view[["Ensemble", "Members", "Test accuracy", "Correct", "Training time (s)", "Inference time (s)"]]
        .style.format({"Test accuracy": "{:.2%}", "Training time (s)": "{:.3f}", "Inference time (s)": "{:.3f}"}))

Matplotlib is building the font cache; this may take a moment.


,Ensemble,Members,Test accuracy,Correct,Training time (s),Inference time (s)
0,Five S=1 models,5,94.32%,83,0.524,0.781
1,"Mixed S=1,1,2,3,4",5,100.00%,88,0.409,0.586


In [4]:
baseline, mixed = results.iloc[0], results.iloc[1]
print(f"Mixed-scale accuracy gain: {100 * (mixed.test_accuracy - baseline.test_accuracy):.2f} percentage points")
print(f"Additional correct predictions: {mixed.correct - baseline.correct} / {len(y_test)}")
print(f"Training speedup: {baseline.training_s / mixed.training_s:.2f}x")
print(f"Inference speedup: {baseline.inference_s / mixed.inference_s:.2f}x")

Mixed-scale accuracy gain: 5.68 percentage points
Additional correct predictions: 5 / 88
Training speedup: 1.28x
Inference speedup: 1.33x


## Why mixing scales helps here

Changing S gives each member a different view of coarse position and temporal scale. The mixed ensemble can combine useful predictions from these complementary views while keeping the same number of members and the same per-member feature budget.

This is a favorable example selected from the paper's evaluated datasets. For a new task, compare a fixed mixed-scale ensemble with the baseline on validation data. The raw S=1 members bypass ROMAN; the routed members use training-fitted channel normalization, matching the paper's recorded MiniRocket protocol.

In [5]:
environment()

{'python': '3.12.14',
 'platform': 'macOS-27.2-arm64-arm-64bit',
 'machine': 'arm64',
 'threads': 1,
 'packages': {'numpy': '2.3.5',
  'pandas': '2.3.3',
  'scipy': '1.17.1',
  'scikit-learn': '1.7.2',
  'sktime': '1.1.0',
  'numba': '0.63.1',
  'threadpoolctl': '3.6.0'}}